In [1]:
import pandas as pd
import numpy as np

data = {
    "OrderID": range(1, 21),
    "CustomerID": np.random.randint(100, 200, 20),
    "ProductCategory": ["Electronics", "Clothing", "Electronics", "Furniture"] * 5,
    "SalesAmount": [1000, 2000, None, 1500, None, 3000, 2500, None, 1800, None,
                    2200, None, 2700, 2900, None, 3100, 3300, None, 3500, None],
    "Region": ["South", "North", "South", "East"] * 5,
    "OrderDate": pd.date_range(start="2023-01-01", periods=20)
}

df = pd.DataFrame(data)

df.to_csv("retail_sales.csv", index=False)

df.head()

,OrderID,CustomerID,ProductCategory,SalesAmount,Region,OrderDate
0,1,198,Electronics,1000.0,South,2023-01-01
1,2,180,Clothing,2000.0,North,2023-01-02
2,3,187,Electronics,NaN,South,2023-01-03
3,4,155,Furniture,1500.0,East,2023-01-04
4,5,146,Electronics,NaN,South,2023-01-05


# New Section

In [2]:
df = pd.read_csv("retail_sales.csv")

df["Imputation_Method"] = None

regional_median = df.groupby(["ProductCategory", "Region"])["SalesAmount"].median()
category_median = df.groupby("ProductCategory")["SalesAmount"].median()

def fill_missing(row):
    if pd.isna(row["SalesAmount"]):
        key = (row["ProductCategory"], row["Region"])

        if key in regional_median and not pd.isna(regional_median[key]):
            row["SalesAmount"] = regional_median[key]
            row["Imputation_Method"] = "Regional_Median"
        else:
            row["SalesAmount"] = category_median[row["ProductCategory"]]
            row["Imputation_Method"] = "Category_Median"

    return row

df = df.apply(fill_missing, axis=1)

df.to_csv("cleaned_retail_sales.csv", index=False)

print(df["Imputation_Method"].value_counts())

df.head()

Imputation_Method
Regional_Median    8
Name: count, dtype: int64


,OrderID,CustomerID,ProductCategory,SalesAmount,Region,OrderDate,Imputation_Method
0,1,198,Electronics,1000.0,South,2023-01-01,None
1,2,180,Clothing,2000.0,North,2023-01-02,None
2,3,187,Electronics,2500.0,South,2023-01-03,Regional_Median
3,4,155,Furniture,1500.0,East,2023-01-04,None
4,5,146,Electronics,2500.0,South,2023-01-05,Regional_Median
